# Analysis — is the learned variance better than a constant variance? (no GPU needed)
Needs only the arrays already saved on Drive by the finished runs. Runs in about a minute.
Set the CPU runtime (Runtime → Change runtime type → CPU) to save GPU quota.
Output: `uq_comparison_<dataset>_seed<seed>.md/.json` in `proposal_runs/analysis` on Drive.

## 1. Mount Google Drive (a pop-up will ask for permission)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the scripts from GitHub (no environment build needed; only numpy)

In [ ]:
import os
%cd /content
if os.path.isdir("/content/proposal/.git"):
    !cd /content/proposal && git pull
else:
    !git clone https://github.com/hrfpour/proposal.git
!ls /content/proposal/code/scripts

## 3. Choose the runs to compare
`PROB_MODELS`: folders under `proposal_ckpt` of the probabilistic runs (the first one is the reference). Include `AGCRN_PROB_DROP10` only after that run exists.
The deterministic `DET_MODEL` run is used for the "constant sigma" rows.

In [ ]:
import json
DATASET   = "PEMS04"
SEED      = 42
PROB_MODELS = ["AGCRN_PROB"]            # later: ["AGCRN_PROB", "AGCRN_PROB_DROP10"]
DET_MODEL   = "AGCRN"
CKPT_ROOT   = "/content/drive/MyDrive/proposal_ckpt"
OUT_DIR     = "/content/drive/MyDrive/proposal_runs/analysis"
NODES = json.load(open(f"/content/drive/MyDrive/proposal_data/{DATASET}/meta.json"))["num_vars"]
print(DATASET, "nodes:", NODES)
for m in PROB_MODELS + [DET_MODEL]:
    !ls {CKPT_ROOT}/{m}/{DATASET}/seed{SEED} | head -3

## 4. Run the comparison

In [ ]:
probs = " ".join(f"--prob {m}" for m in PROB_MODELS)
!python /content/proposal/code/scripts/compare_gaussian_baselines.py --ckpt-root {CKPT_ROOT} --dataset {DATASET} --seed {SEED} --num-nodes {NODES} {probs} --det {DET_MODEL} --out-dir {OUT_DIR}